# 🏃 애플 워치 운동 기록 분석
판다스를 활용하여 애플 워치 운동 데이터를 모델링하고 분석합니다.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

## 1. 샘플 데이터 생성
애플 워치에서 수출되는 운동 기록 형식으로 샘플 데이터를 생성합니다.

In [ ]:
random.seed(42)
np.random.seed(42)

workout_types = ['달리기', '걷기', '사이클링', '수영', '등산', 'HIIT', '요가', '줄넘기']

n = 120  # 4개월치 데이터

start_date = datetime(2025, 11, 1)
dates = [start_date + timedelta(days=i) for i in sorted(random.sample(range(120), n))]

workout_type_list = random.choices(workout_types, weights=[30, 20, 15, 10, 10, 7, 5, 3], k=n)

# 운동 종류별 기본 칼로리, 심박수, 시간 범위 설정
workout_params = {
    '달리기':   {'cal': (300, 600), 'hr': (140, 175), 'min': (20, 60)},
    '걷기':     {'cal': (100, 300), 'hr': (90, 120),  'min': (30, 90)},
    '사이클링': {'cal': (250, 550), 'hr': (120, 160), 'min': (30, 90)},
    '수영':     {'cal': (200, 500), 'hr': (120, 155), 'min': (20, 60)},
    '등산':     {'cal': (400, 800), 'hr': (130, 165), 'min': (60, 180)},
    'HIIT':    {'cal': (300, 500), 'hr': (150, 185), 'min': (20, 45)},
    '요가':     {'cal': (80, 200),  'hr': (70, 110),  'min': (30, 60)},
    '줄넘기':   {'cal': (200, 400), 'hr': (140, 170), 'min': (10, 30)},
}

calories, avg_heart_rate, duration_min, distance_km = [], [], [], []

for wtype in workout_type_list:
    p = workout_params[wtype]
    cal = round(random.uniform(*p['cal']), 1)
    hr  = round(random.uniform(*p['hr']), 1)
    mins = round(random.uniform(*p['min']), 1)
    
    if wtype in ['달리기', '걷기', '사이클링', '등산']:
        speed = {'달리기': (6, 12), '걷기': (3, 6), '사이클링': (15, 30), '등산': (2, 5)}
        dist = round(mins / 60 * random.uniform(*speed[wtype]), 2)
    else:
        dist = None
    
    calories.append(cal)
    avg_heart_rate.append(hr)
    duration_min.append(mins)
    distance_km.append(dist)

df = pd.DataFrame({
    'date':           dates,
    'workout_type':   workout_type_list,
    'duration_min':   duration_min,
    'calories':       calories,
    'avg_heart_rate': avg_heart_rate,
    'distance_km':    distance_km,
})

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'총 운동 기록 수: {len(df)}건')
df.head(10)

## 2. 기본 정보 확인

In [ ]:
print('=== 데이터 타입 및 결측치 ===')  
df.info()
print()
print('=== 기술 통계 ===')
df.describe().round(2)

## 3. 운동 종류별 분석

In [ ]:
workout_summary = df.groupby('workout_type').agg(
    총_횟수       = ('workout_type', 'count'),
    평균_시간_분  = ('duration_min',   'mean'),
    총_칼로리     = ('calories',        'sum'),
    평균_칼로리   = ('calories',        'mean'),
    평균_심박수   = ('avg_heart_rate',  'mean'),
    평균_거리_km  = ('distance_km',     'mean'),
).round(2)

workout_summary = workout_summary.sort_values('총_횟수', ascending=False)
print('=== 운동 종류별 요약 ===')
workout_summary

## 4. 월별 트렌드 분석

In [ ]:
df['year_month'] = df['date'].dt.to_period('M')

monthly = df.groupby('year_month').agg(
    운동_횟수   = ('workout_type', 'count'),
    총_시간_분  = ('duration_min',  'sum'),
    총_칼로리   = ('calories',       'sum'),
    평균_심박수 = ('avg_heart_rate', 'mean'),
).round(2)

monthly['총_시간_시간'] = (monthly['총_시간_분'] / 60).round(1)

print('=== 월별 운동 트렌드 ===')
monthly

## 5. 요일별 운동 패턴

In [ ]:
day_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df['weekday'] = df['date'].dt.dayofweek.map(day_map)

weekday_order = ['월', '화', '수', '목', '금', '토', '일']

weekday_stats = df.groupby('weekday').agg(
    운동_횟수 = ('workout_type', 'count'),
    평균_칼로리 = ('calories', 'mean'),
    평균_시간 = ('duration_min', 'mean'),
).round(2)

weekday_stats = weekday_stats.reindex(weekday_order)

print('=== 요일별 운동 패턴 ===')
weekday_stats

## 6. 심박수 구간별 분류 (운동 강도)

In [ ]:
def classify_intensity(hr):
    if hr < 100:
        return '저강도 (회복)'
    elif hr < 130:
        return '중저강도 (지방연소)'
    elif hr < 155:
        return '중강도 (유산소)'
    elif hr < 170:
        return '고강도 (무산소)'
    else:
        return '최고강도 (최대심박수)'

df['intensity'] = df['avg_heart_rate'].apply(classify_intensity)

intensity_stats = df.groupby('intensity').agg(
    횟수       = ('workout_type', 'count'),
    평균_칼로리 = ('calories', 'mean'),
    평균_시간  = ('duration_min', 'mean'),
).round(2)

intensity_order = ['저강도 (회복)', '중저강도 (지방연소)', '중강도 (유산소)', '고강도 (무산소)', '최고강도 (최대심박수)']
intensity_stats = intensity_stats.reindex([x for x in intensity_order if x in intensity_stats.index])

print('=== 운동 강도별 분포 ===')
intensity_stats

## 7. 칼로리 효율 분석 (칼로리/분)

In [ ]:
df['cal_per_min'] = (df['calories'] / df['duration_min']).round(2)

cal_efficiency = df.groupby('workout_type')['cal_per_min'].agg(['mean', 'max', 'min']).round(2)
cal_efficiency.columns = ['평균_칼로리/분', '최대_칼로리/분', '최소_칼로리/분']
cal_efficiency = cal_efficiency.sort_values('평균_칼로리/분', ascending=False)

print('=== 운동 종류별 칼로리 효율 (칼로리/분) ===')
cal_efficiency

## 8. 개인 기록 (베스트 기록)

In [ ]:
print('=== 개인 최고 기록 ===\n')

best_cal = df.loc[df['calories'].idxmax()]
print(f'최고 칼로리 소모: {best_cal["calories"]} kcal')
print(f'  - 운동: {best_cal["workout_type"]} | 날짜: {best_cal["date"].date()} | 시간: {best_cal["duration_min"]}분\n')

best_dur = df.loc[df['duration_min'].idxmax()]
print(f'최장 운동 시간: {best_dur["duration_min"]}분 ({best_dur["duration_min"]/60:.1f}시간)')
print(f'  - 운동: {best_dur["workout_type"]} | 날짜: {best_dur["date"].date()}\n')

best_hr = df.loc[df['avg_heart_rate'].idxmax()]
print(f'최고 평균 심박수: {best_hr["avg_heart_rate"]} bpm')
print(f'  - 운동: {best_hr["workout_type"]} | 날짜: {best_hr["date"].date()}\n')

running_df = df[df['workout_type'] == '달리기'].dropna(subset=['distance_km'])
if not running_df.empty:
    best_run = running_df.loc[running_df['distance_km'].idxmax()]
    print(f'달리기 최장 거리: {best_run["distance_km"]} km')
    print(f'  - 날짜: {best_run["date"].date()} | 시간: {best_run["duration_min"]}분')

## 9. 전체 누적 통계 요약

In [ ]:
print('=' * 40)
print('     애플 워치 운동 기록 종합 요약')
print('=' * 40)
print(f'  분석 기간     : {df["date"].min().date()} ~ {df["date"].max().date()}')
print(f'  총 운동 횟수  : {len(df)}회')
print(f'  총 운동 시간  : {df["duration_min"].sum():.0f}분 ({df["duration_min"].sum()/60:.1f}시간)')
print(f'  총 칼로리 소모: {df["calories"].sum():.0f} kcal')
print(f'  평균 심박수   : {df["avg_heart_rate"].mean():.1f} bpm')
dist_total = df['distance_km'].sum()
print(f'  총 이동 거리  : {dist_total:.1f} km')
print(f'  가장 많은 운동: {df["workout_type"].value_counts().idxmax()}')
print('=' * 40)

## 10. CSV로 저장

In [ ]:
df.to_csv('apple_watch_workout_data.csv', index=False, encoding='utf-8-sig')
print('데이터가 apple_watch_workout_data.csv 파일로 저장되었습니다.')
df[['date', 'workout_type', 'duration_min', 'calories', 'avg_heart_rate', 'distance_km', 'intensity', 'cal_per_min']].tail(10)

---
## 11. 주간 이동 거리 분석

In [ ]:
# 주차 컬럼 추가 (ISO 기준 연도-주차)
df['week'] = df['date'].dt.to_period('W')

# 야외 운동(거리 있는 종목)만 필터링
outdoor_df = df.dropna(subset=['distance_km']).copy()

weekly_distance = outdoor_df.groupby(['week', 'workout_type'])['distance_km'].sum().reset_index()
weekly_distance.columns = ['주차', '운동종류', '총_거리_km']

# 주차별 전체 합산
weekly_total = outdoor_df.groupby('week').agg(
    총_이동거리_km = ('distance_km', 'sum'),
    운동_횟수     = ('workout_type', 'count'),
    총_칼로리     = ('calories', 'sum'),
).round(2).reset_index()
weekly_total.columns = ['주차', '총_이동거리_km', '운동_횟수', '총_칼로리']

print('=== 주간 이동 거리 요약 ===')
weekly_total

## 12. GPS 경로 시뮬레이션 데이터 생성
서울 주요 운동 장소(한강, 올림픽공원, 남산 등)를 기반으로 GPS 경로를 시뮬레이션합니다.

In [ ]:
# 서울 주요 운동 코스 중심 좌표
locations = {
    '달리기': [
        {'name': '한강공원 (여의도)',  'lat': 37.5283, 'lon': 126.9326, 'radius': 0.018},
        {'name': '한강공원 (반포)',    'lat': 37.5120, 'lon': 126.9954, 'radius': 0.015},
        {'name': '올림픽공원',         'lat': 37.5205, 'lon': 127.1220, 'radius': 0.016},
    ],
    '걷기': [
        {'name': '북서울꿈의숲',       'lat': 37.6243, 'lon': 127.0371, 'radius': 0.012},
        {'name': '서울숲',             'lat': 37.5444, 'lon': 127.0374, 'radius': 0.010},
        {'name': '월드컵공원',         'lat': 37.5702, 'lon': 126.8976, 'radius': 0.013},
    ],
    '사이클링': [
        {'name': '한강 자전거도로',    'lat': 37.5340, 'lon': 126.9700, 'radius': 0.030},
        {'name': '탄천 자전거도로',    'lat': 37.4700, 'lon': 127.1000, 'radius': 0.025},
    ],
    '등산': [
        {'name': '남산',               'lat': 37.5512, 'lon': 126.9882, 'radius': 0.020},
        {'name': '북한산',             'lat': 37.6587, 'lon': 126.9769, 'radius': 0.035},
        {'name': '관악산',             'lat': 37.4441, 'lon': 126.9646, 'radius': 0.030},
    ],
}

def simulate_gps_route(center_lat, center_lon, distance_km, n_points=50):
    """중심 좌표에서 거리 비례 반경으로 GPS 경로 시뮬레이션"""
    # 1km ≈ 0.009도 (위도), 0.011도 (경도)
    radius_lat = min(distance_km * 0.004, 0.04)
    radius_lon = min(distance_km * 0.005, 0.05)
    
    angles = np.linspace(0, 2 * np.pi, n_points)
    noise_lat = np.random.normal(0, radius_lat * 0.15, n_points)
    noise_lon = np.random.normal(0, radius_lon * 0.15, n_points)
    
    lats = center_lat + radius_lat * np.sin(angles) + noise_lat
    lons = center_lon + radius_lon * np.cos(angles) + noise_lon
    
    return list(zip(lats.round(6), lons.round(6)))

# GPS 경로 데이터프레임 생성
gps_records = []

for _, row in outdoor_df.iterrows():
    wtype = row['workout_type']
    if wtype not in locations:
        continue
    
    loc = random.choice(locations[wtype])
    route = simulate_gps_route(loc['lat'], loc['lon'], row['distance_km'])
    
    gps_records.append({
        'date':         row['date'],
        'week':         row['week'],
        'workout_type': wtype,
        'location':     loc['name'],
        'distance_km':  row['distance_km'],
        'calories':     row['calories'],
        'route':        route,
        'start_lat':    route[0][0],
        'start_lon':    route[0][1],
    })

gps_df = pd.DataFrame(gps_records)
print(f'GPS 경로 데이터: {len(gps_df)}건')
gps_df[['date', 'workout_type', 'location', 'distance_km']].head(10)

## 13. 주간 이동 경로 지도 시각화 (Folium)
> `pip install folium` 이 필요합니다.

In [ ]:
try:
    import folium
    from folium.plugins import MarkerCluster
    FOLIUM_AVAILABLE = True
    print('folium 사용 가능')
except ImportError:
    FOLIUM_AVAILABLE = False
    print('folium 미설치 → pip install folium 실행 후 재시도')

In [ ]:
if FOLIUM_AVAILABLE:
    # 운동 종류별 경로 색상
    color_map = {
        '달리기':   '#E74C3C',  # 빨강
        '걷기':     '#2ECC71',  # 초록
        '사이클링': '#3498DB',  # 파랑
        '등산':     '#9B59B6',  # 보라
    }

    # ── 전체 기간 운동 경로 지도 ──────────────────────────────────────
    m_all = folium.Map(location=[37.5340, 126.9700], zoom_start=11, tiles='CartoDB positron')

    for _, row in gps_df.iterrows():
        color = color_map.get(row['workout_type'], '#888888')
        folium.PolyLine(
            locations=row['route'],
            color=color,
            weight=2.5,
            opacity=0.6,
            tooltip=f"{row['date'].strftime('%Y-%m-%d')} {row['workout_type']} "
                    f"| {row['location']} | {row['distance_km']}km | {row['calories']}kcal"
        ).add_to(m_all)
        # 시작 마커
        folium.CircleMarker(
            location=[row['start_lat'], row['start_lon']],
            radius=4, color=color, fill=True, fill_opacity=0.8,
            tooltip=f"{row['workout_type']} ({row['date'].strftime('%m/%d')})"
        ).add_to(m_all)

    # 범례 HTML
    legend_html = '''
    <div style="position:fixed;bottom:30px;left:30px;z-index:1000;
                background:white;padding:12px;border-radius:8px;
                box-shadow:2px 2px 6px rgba(0,0,0,0.3);font-size:13px;">
        <b>운동 경로</b><br>
        <span style="color:#E74C3C">●</span> 달리기<br>
        <span style="color:#2ECC71">●</span> 걷기<br>
        <span style="color:#3498DB">●</span> 사이클링<br>
        <span style="color:#9B59B6">●</span> 등산
    </div>'''
    m_all.get_root().html.add_child(folium.Element(legend_html))

    m_all.save('workout_map_all.html')
    print('전체 운동 경로 지도 저장: workout_map_all.html')
    m_all
else:
    print('folium 미설치 상태 - 지도를 표시하려면 pip install folium 후 재실행')

## 14. 특정 주간 운동 경로 지도

In [ ]:
if FOLIUM_AVAILABLE:
    # 운동 기록이 가장 많은 주 선택
    top_week = gps_df.groupby('week').size().idxmax()
    week_df = gps_df[gps_df['week'] == top_week]

    print(f'선택된 주: {top_week}  (운동 {len(week_df)}회)')

    m_week = folium.Map(location=[37.5340, 126.9700], zoom_start=12, tiles='OpenStreetMap')

    for idx, row in week_df.iterrows():
        color = color_map.get(row['workout_type'], '#888888')
        folium.PolyLine(
            locations=row['route'],
            color=color, weight=4, opacity=0.8,
            tooltip=f"{row['date'].strftime('%Y-%m-%d')} {row['workout_type']} "
                    f"| {row['location']}\n거리: {row['distance_km']}km | 칼로리: {row['calories']}kcal"
        ).add_to(m_week)

        folium.Marker(
            location=[row['start_lat'], row['start_lon']],
            popup=folium.Popup(
                f"<b>{row['workout_type']}</b><br>"
                f"날짜: {row['date'].strftime('%Y-%m-%d')}<br>"
                f"장소: {row['location']}<br>"
                f"거리: {row['distance_km']} km<br>"
                f"칼로리: {row['calories']} kcal",
                max_width=200
            ),
            icon=folium.Icon(color={
                '달리기':'red','걷기':'green','사이클링':'blue','등산':'purple'
            }.get(row['workout_type'], 'gray'), icon='info-sign')
        ).add_to(m_week)

    # 주간 통계 오버레이
    week_stats = week_df.agg({'distance_km':'sum','calories':'sum'})
    stats_html = f'''
    <div style="position:fixed;top:10px;right:10px;z-index:1000;
                background:white;padding:12px;border-radius:8px;
                box-shadow:2px 2px 6px rgba(0,0,0,0.3);font-size:13px;">
        <b>주간 통계 ({top_week})</b><br>
        운동 횟수: {len(week_df)}회<br>
        총 이동거리: {week_stats["distance_km"]:.1f} km<br>
        총 칼로리: {week_stats["calories"]:.0f} kcal
    </div>'''
    m_week.get_root().html.add_child(folium.Element(stats_html))

    m_week.save('workout_map_weekly.html')
    print(f'주간 운동 경로 지도 저장: workout_map_weekly.html')
    m_week
else:
    print('folium 미설치 상태')

## 15. 주간 이동 거리 막대 그래프 (텍스트 시각화)

In [ ]:
print('=== 주간 이동 거리 (km) ===')
print(f'{"주차":<20} {"거리":>8} {"막대"}') 
print('-' * 55)

for _, row in weekly_total.iterrows():
    bar_len = int(row['총_이동거리_km'] / 2)
    bar = '█' * bar_len
    print(f'{str(row["주차"]):<20} {row["총_이동거리_km"]:>6.1f}km  {bar}')

print()
print(f'주간 평균 이동 거리: {weekly_total["총_이동거리_km"].mean():.1f} km')
print(f'최대 주간 이동 거리: {weekly_total["총_이동거리_km"].max():.1f} km  ({weekly_total.loc[weekly_total["총_이동거리_km"].idxmax(), "주차"]})')